In [ ]:
import matplotlib.pyplot as plt
from itertools import combinations

from test.test_filters import make_test_instrument
from test.utilities import compile_and_scan, acceptable_total_counts

In [ ]:

instr = make_test_instrument()
with open("Test_filters.instr", "w") as f:
    instr.to_file(f)

filters = {
    "Filter_gen": 2183.87,
    "NCrystal_sample": 1875.02,
    "PowderN": 1868.5,
    "Radial_filter_collimator": 1875.02,
}
results = compile_and_scan(instr, {'filter': '0:3'}, ncount=1e6, seed=1, use_temp_dir=True)
filternames = list(filters.keys())
l_out = {filternames[i]: res['L_out'] for i, res in enumerate(results['scan_result'])}

for name, expected in filters.items():
    assert acceptable_total_counts(l_out[name], expected, tolerance=1.0), (
        f"Total counts for {name} differ from expected by more than 10%: "
        f"{l_out[name]['I'].sum()} vs {expected}"
    )

In [ ]:
def is_acceptable(collimator, dat):
    return abs(dat['I'].sum() / filters[collimator] - 1) < 0.04

In [ ]:
fig, ax = plt.subplots(len(filters), len(filters), sharex=True, sharey=True)
fig.set_size_inches(13*4/3, 8*4/3)
for i, (c, v) in enumerate(l_out.items()):
    ax[0,i].plot(v['L'], v['I'], label=c)
    ax[0,i].legend()
    plt.setp(ax[0,i], facecolor=(0,1,0,0.1) if is_acceptable(c, v) else (1,0,0,0.1))

for (i, f), (j, s) in combinations(enumerate(filters), 2):
    x = l_out[f]['L']
    y = l_out[f]['I'] - l_out[s]['I']
    t = l_out[f]['I'].sum()
    ax[j, i].plot(x, y, label=f'{f}-{s}')
    ax[j, i].text(7, 10, rf'$\Delta$ = {100 * abs(sum(y)/t): 4.1f}%')
    ax[j, i].legend()
